# TransferBench

# Experiment 5

## Vision Transformer (ViT-B16)

### Feature Extraction

This notebook evaluates Vision Transformer (ViT-B16) using transfer learning.

Only the classification head is trainable.

Outputs

- Trained Model
- Evaluation Metrics
- Learning Curves
- Confusion Matrix

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

C:\Users\pc1\TransferBench


In [2]:
import torch
import torch.nn as nn

from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

from utils.dataset import load_dataset

from utils.models import build_model

from utils.training import (
    train_model,
    print_summary,
)

from utils.evaluation import (
    calculate_metrics,
    print_metrics,
    export_metrics_csv,
    build_confusion_matrix,
)

from utils.visualization import (
    plot_loss_curve,
    plot_accuracy_curve,
    plot_confusion_matrix,
)

from utils.paths import *

Import OK ✅


In [3]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


## Load Dataset

In [4]:
train_loader, test_loader, class_names = load_dataset(
    dataset_name="cifar10",
    batch_size=32,
    image_size=224,
)

C:\Users\pc1\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


## Build Vision Transformer

In [5]:
model = build_model(
    model_name="vit",
    num_classes=10,
    strategy="freeze",
)

model = model.to(device)

print(model)

VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine

## Optimizer

In [6]:
criterion = nn.CrossEntropyLoss()

optimizer = Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters(),
    ),
    lr=1e-3,
)

optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)

In [7]:
scheduler = ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)

In [8]:
EPOCHS = 20

checkpoint_path = get_model_path(
    "best_vit_freeze"
)

print(checkpoint_path)

C:\Users\pc1\TransferBench\models\best_vit_freeze.pth


## Train Model

In [9]:
results = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=EPOCHS,
    device=device,
    checkpoint_path=checkpoint_path,
)

Training Started
Epoch 01/20 | Train Loss: 0.2073 | Train Acc: 0.9368 | Val Loss: 0.1556 | Val Acc: 0.9480
Epoch 02/20 | Train Loss: 0.1374 | Train Acc: 0.9547 | Val Loss: 0.1444 | Val Acc: 0.9519
Epoch 03/20 | Train Loss: 0.1244 | Train Acc: 0.9578 | Val Loss: 0.1389 | Val Acc: 0.9559
Epoch 04/20 | Train Loss: 0.1176 | Train Acc: 0.9609 | Val Loss: 0.1416 | Val Acc: 0.9520
Epoch 05/20 | Train Loss: 0.1113 | Train Acc: 0.9626 | Val Loss: 0.1425 | Val Acc: 0.9511
Epoch 06/20 | Train Loss: 0.1063 | Train Acc: 0.9644 | Val Loss: 0.1425 | Val Acc: 0.9531
Epoch 07/20 | Train Loss: 0.0955 | Train Acc: 0.9680 | Val Loss: 0.1346 | Val Acc: 0.9549
Epoch 08/20 | Train Loss: 0.0944 | Train Acc: 0.9682 | Val Loss: 0.1335 | Val Acc: 0.9546
Epoch 09/20 | Train Loss: 0.0910 | Train Acc: 0.9683 | Val Loss: 0.1341 | Val Acc: 0.9553
Epoch 10/20 | Train Loss: 0.0851 | Train Acc: 0.9710 | Val Loss: 0.1342 | Val Acc: 0.9556
Epoch 11/20 | Train Loss: 0.0868 | Train Acc: 0.9701 | Val Loss: 0.1340 | Val Acc: 

## Evaluation

In [ ]:
metrics = calculate_metrics(
    results["targets"],
    results["predictions"],
)

print_metrics(metrics)

In [ ]:
metrics_df = export_metrics_csv(
    metrics=metrics,
    model_name="ViT-B16",
    strategy="Feature Extraction",
    training_time=results["training_time"],
    save_path=get_metrics_path("vit_freeze"),
)

metrics_df

## Learning Curves

In [ ]:
plot_loss_curve(
    history=results["history"],
    save_path=get_loss_plot_path("vit_freeze"),
)

In [ ]:
plot_accuracy_curve(
    history=results["history"],
    save_path=get_accuracy_plot_path("vit_freeze"),
)

## Confusion Matrix

In [ ]:
cm = build_confusion_matrix(
    results["targets"],
    results["predictions"],
)

plot_confusion_matrix(
    matrix=cm,
    class_names=class_names,
    save_path=get_confusion_matrix_path("vit_freeze"),
)

## Training Summary

In [ ]:
print_summary(results)

In [ ]:
metrics_df

In [ ]:
results["history"]

In [ ]:
results

## Generated Files

In [ ]:
print("=" * 60)

print("Saved Model")
print(get_model_path("best_vit_freeze"))

print()

print("Metrics CSV")
print(get_metrics_path("vit_freeze"))

print()

print("Loss Curve")
print(get_loss_plot_path("vit_freeze"))

print()

print("Accuracy Curve")
print(get_accuracy_plot_path("vit_freeze"))

print()

print("Confusion Matrix")
print(get_confusion_matrix_path("vit_freeze"))

print("=" * 60)

In [ ]:
from pathlib import Path

print("========== MODELS ==========")
for f in sorted(Path(MODEL_DIR).glob("*")):
    print(f.name)

print("\n========== FIGURES ==========")
for f in sorted(Path(FIGURE_DIR).glob("*")):
    print(f.name)

print("\n========== RESULTS ==========")
for f in sorted(Path(RESULT_DIR).glob("*")):
    print(f.name)